# Volume 8. Toy Prediction And Signal Map

Question: what is the difference between a toy overlap readout and a research claim about ERP-like signals?

This notebook runs a tiny next-token demo, then points to the research files that carry stronger signal claims.

In [ ]:
from pathlib import Path

import pandas as pd

from neural_assemblies.assembly_calculus import (
    build_next_token_model,
    predict_next_token,
    score_corpus,
    train_on_corpus,
)
from neural_assemblies.core.brain import Brain


In [ ]:
vocab = ["the", "cat", "dog", "runs", "sleeps"]
stimuli = {word: f"stim_{word}" for word in vocab}
corpus = [["the", "cat", "runs"], ["the", "dog", "sleeps"]]

brain = Brain(p=0.05, save_winners=True, seed=29, engine="numpy_sparse")
for stimulus in stimuli.values():
    brain.add_stimulus(stimulus, size=35)
brain.add_area("LEX", n=2_000, k=35, beta=0.08)

lexicon = build_next_token_model(brain, "LEX", vocab, stimuli, rounds=5)
train_on_corpus(brain, "LEX", corpus, stimuli, rounds_per_token=4, repetitions=1)

pd.DataFrame(predict_next_token(brain, "LEX", ["the", "cat"], stimuli, lexicon, rounds_per_token=4), columns=["word", "overlap"]).head()


In [ ]:
pd.DataFrame([score_corpus(brain, "LEX", corpus, stimuli, lexicon, rounds_per_token=4)])


In [ ]:
repo_root = Path.cwd()
while not (repo_root / "research").exists():
    if repo_root.parent == repo_root:
        raise RuntimeError("Could not locate repository root")
    repo_root = repo_root.parent

paths = [
    "research/core_questions/Q22_n400_global_energy",
    "research/claims/N400_GLOBAL_ENERGY.md",
    "research/plans/N400_MATHEMATICAL_ANALYSIS.md",
    "research/plans/P600_REANALYSIS.md",
]
pd.DataFrame([{"path": path, "exists": (repo_root / path).exists()} for path in paths])
